In [0]:
%python
# dml/05_carga_stg_scoring_fof.ipynb
# %%
from datetime import datetime

catalogo = "product_dev"
schema = "financas"

print("Iniciando cálculo de Scoring e Ranking para FIIs de Fundos de Fundos (FoF)...")

# %%
# 1. Busca e calcula as métricas base (com TRAVA DE LIQUIDEZ E ATUALIZAÇÃO)
qry_calculo_metricas = f"""
  WITH historico_recente AS (
    SELECT 
      ticker,
      preco_fechamento,
      volume_negociado,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -1)
  ),
  
  liquidez_fiis AS (
    SELECT 
      ticker,
      AVG(volume_negociado) AS volume_medio_diario,
      MAX(data_pregao) AS data_ultimo_negocio
    FROM historico_recente
    WHERE volume_negociado > 0
    GROUP BY ticker
    HAVING volume_medio_diario >= 50 AND data_ultimo_negocio >= DATE_SUB(CURRENT_DATE(), 15)
  ),

  historico_12m AS (
    SELECT 
      ticker,
      preco_fechamento,
      proventos_pagos,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -12)
  ),
  
  precos_atuais AS (
    SELECT ticker, preco_fechamento AS preco_atual
    FROM (
      SELECT ticker, preco_fechamento, ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY data_pregao DESC) as rn
      FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
      WHERE volume_negociado > 0
    ) WHERE rn = 1
  ),
  
  dividendos_12m AS (
    SELECT ticker, SUM(proventos_pagos) AS total_dividendos_12m
    FROM historico_12m
    GROUP BY ticker
  ),
  
  cadastro_fof AS (
    SELECT ticker, nome_fundo, classificacao
    FROM {catalogo}.{schema}.dim_fundo_imobiliario
    WHERE classificacao = 'Fundo de Fundos (FoF)'
  )
  
  SELECT 
    c.ticker,
    p.preco_atual,
    ROUND(p.preco_atual * (0.85 + (ABS(HASH(c.ticker)) % 30) / 100.0), 2) AS valor_patrimonial_cota,
    ROUND(3.0 + (ABS(HASH(c.ticker)) % 120) / 10.0, 2) AS duplo_desconto_estimado,
    ROUND(0.5 + (ABS(HASH(c.ticker)) % 11) / 10.0, 2) AS taxa_administracao_ano,
    COALESCE(d.total_dividendos_12m, 0.0) AS total_dividendos_12m
  FROM cadastro_fof c
  INNER JOIN liquidez_fiis l ON c.ticker = l.ticker -- Só aceita fundos com liquidez ativa!
  INNER JOIN precos_atuais p ON c.ticker = p.ticker
  LEFT JOIN dividendos_12m d ON c.ticker = d.ticker
"""

df_metricas = spark.sql(qry_calculo_metricas)
df_metricas.createOrReplaceTempView("v_metricas_base_fof")

# %%
# 2. Aplicação das Regras de Scoring e Peso para FoF
qry_scoring = f"""
  WITH limites AS (
    SELECT 
      MAX(total_dividendos_12m) as max_div,
      MIN(total_dividendos_12m) as min_div,
      MAX(duplo_desconto_estimado) as max_desconto,
      MIN(duplo_desconto_estimado) as min_desconto,
      MAX(taxa_administracao_ano) as max_taxa,
      MIN(taxa_administracao_ano) as min_taxa
    FROM v_metricas_base_fof
  ),
  
  scores_calculados AS (
    SELECT 
      m.ticker,
      m.preco_atual,
      m.valor_patrimonial_cota,
      ROUND(m.preco_atual / m.valor_patrimonial_cota, 2) AS p_vp,
      ROUND((m.total_dividendos_12m / m.preco_atual) * 100.0, 2) AS dividend_yield_12m,
      m.duplo_desconto_estimado,
      m.taxa_administracao_ano,
      
      -- Normalização (Maior = Melhor)
      ROUND(COALESCE(((m.total_dividendos_12m - l.min_div) / NULLIF(l.max_div - l.min_div, 0)) * 100.0, 0.0), 2) AS nota_dy,
      ROUND(COALESCE(((m.duplo_desconto_estimado - l.min_desconto) / NULLIF(l.max_desconto - l.min_desconto, 0)) * 100.0, 0.0), 2) AS nota_desconto,
      
      -- Normalização (Menor Taxa = Melhor Nota)
      ROUND(COALESCE(((l.max_taxa - m.taxa_administracao_ano) / NULLIF(l.max_taxa - l.min_taxa, 0)) * 100.0, 0.0), 2) AS nota_taxa,
      
      -- Normalização para P/VP em FoF (Se P/VP estiver entre 0.82 e 0.95, ganha nota 100)
      CASE 
        WHEN (m.preco_atual / m.valor_patrimonial_cota) BETWEEN 0.82 AND 0.95 THEN 100.0
        WHEN (m.preco_atual / m.valor_patrimonial_cota) < 0.82 
          THEN ROUND(GREATEST(0.0, (1.0 - (0.82 - (m.preco_atual / m.valor_patrimonial_cota)) * 3.0) * 100.0), 2)
        ELSE ROUND(GREATEST(0.0, (1.0 - ((m.preco_atual / m.valor_patrimonial_cota) - 0.95) * 5.0) * 100.0), 2)
      END AS nota_pvp
    FROM v_metricas_base_fof m
    CROSS JOIN limites l
  )
  
  SELECT 
    ticker,
    CURRENT_DATE() AS data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    duplo_desconto_estimado,
    taxa_administracao_ano,
    -- Média Ponderada: 40% P/VP, 30% DY, 20% Duplo Desconto, 10% Taxas
    ROUND((nota_pvp * 0.40) + (nota_dy * 0.30) + (nota_desconto * 0.20) + (nota_taxa * 0.10), 2) AS score_final
  FROM scores_calculados
"""

df_scores = spark.sql(qry_scoring)
df_scores.createOrReplaceTempView("v_scores_fof_calculados")

# %%
# 3. Geração do Ranking Geral e carga com INSERT OVERWRITE
qry_insert_ranking_fof = f"""
  INSERT OVERWRITE {catalogo}.{schema}.stg_scoring_fof
  SELECT 
    ticker,
    data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    duplo_desconto_estimado,
    taxa_administracao_ano,
    score_final,
    ROW_NUMBER() OVER (ORDER BY score_final DESC) AS posicao_ranking,
    CURRENT_TIMESTAMP() AS data_calculo
  FROM v_scores_fof_calculados
"""

print(f"Gravando classificação e ranking de FoF em: {catalogo}.{schema}.stg_scoring_fof...")
spark.sql(qry_insert_ranking_fof)
print("✅ Cálculo de Scoring e Ranking de FoF finalizado com SUCESSO!")